### Transformar Datos Job Positions
1. Preprocesar la cadena JSON para corregir los problemas de calidad de los datos
2. Transformar cadena JSON en objeto JSON
3. Escribir los datos "transformados" en el schema "silver" - "job_positions_json"
4. Acceder a los elementos del objeto JSON
5. Eliminar elementos duplicados del array
6. Anular el anidamiento del array
7. Escribir los datos "transformados" en el schema "silver" - "job_positions"

In [0]:
df_job_positions = spark.table("zenviro.bronze.job_positions_py")
display(df_job_positions)

#### 1. Preprocesar la cadena JSON para corregir los problemas de calidad de los datos

In [0]:
from pyspark.sql.functions import regexp_replace

df_fixed_value = (
    df_job_positions
    .select(
        regexp_replace("value", '"title": ([^",}]+)', '"title": "$1"').alias("fixed_value")
    )
)
display(df_fixed_value)

#### 2. Transformar cadena JSON en objeto JSON

In [0]:
from pyspark.sql.functions import schema_of_json, col

get_schema = (
    df_fixed_value
    .select(
        schema_of_json(col("fixed_value")).alias("schema")
    )
)
display(get_schema.limit(1))

In [0]:
job_positions_schema = '''STRUCT<description: STRING, job_position_id: BIGINT, salary: ARRAY<STRUCT<salary_max: BIGINT, salary_min: BIGINT>>, title: STRING>'''

In [0]:
from pyspark.sql.functions import from_json

df_job_positions_json = (
    df_fixed_value
    .select(
        from_json("fixed_value", job_positions_schema).alias("json_value")
    )
)
display(df_job_positions_json)

#### 3. Escribir los datos "transformados" en el schema "silver" - "job_positions_json"

In [0]:
df_job_positions_json.writeTo("zenviro.silver.job_positions_json_py").createOrReplace()

In [0]:
display(spark.table("zenviro.silver.job_positions_json_py"))

#### 4. Acceder a los elementos del objeto JSON
> `<column_name.object>`

In [0]:
df_job_positions_item = (
    df_job_positions_json
    .select (
       "json_value.job_position_id",
       "json_value.title",
       "json_value.description",
       "json_value.salary"
    )
)
display(df_job_positions_item)

#### 5. Eliminar elementos duplicados del array
[Doc - Función "array_distinct"](https://learn.microsoft.com/es-es/azure/databricks/sql/language-manual/functions/array_distinct)

In [0]:
from pyspark.sql.functions import array_distinct

df_job_positions_drop_duplicates = (
    df_job_positions_json
    .select (
       "json_value.job_position_id",
       "json_value.title",
       "json_value.description",
       array_distinct("json_value.salary").alias("salary")
    )
)
display(df_job_positions_drop_duplicates)

#### 6. Anular el anidamiento del array

In [0]:
from pyspark.sql.functions import explode

df_job_positions_exploded = (
    df_job_positions_drop_duplicates
    .select (
        "job_position_id",
        "title",
        "description",
        explode("salary").alias("salary")
    )
)
display(df_job_positions_exploded)

In [0]:
from pyspark.sql.functions import col, cast

df_job_positions_normalized = (
    df_job_positions_exploded
    .select (
        "job_position_id",
        "title",
        "description",
        "salary.salary_min",
        "salary.salary_max"
    )
)
display(df_job_positions_normalized)

#### 7. Escribir los datos "transformados" en el schema "silver" - "job_positions_py"

In [0]:
df_job_positions_normalized.writeTo("zenviro.silver.job_positions_py").createOrReplace()

In [0]:
display(spark.table("zenviro.silver.job_positions_py"))